# Fine-tune a Small LM with Unsloth — local-steps companion notebook

This notebook is a companion to the course's [Fine-tune a Small Language Model with Unsloth](https://github.com/abderrahim-lectures/python-data-analysis-course/tree/main/docs/projects/finetune-llm-unsloth) project.

**Scope — read this first.** This notebook covers only the parts of that project that normally run *locally* with `uv`:

- **Step 1 (dataset prep)** — building a small instruction/response `dataset.jsonl`, mirroring `build_dataset.py`.
- **Step 3 (local inference)** — loading a base model plus a downloaded LoRA adapter and running a prompt, mirroring `infer.py`.

**It does NOT include the fine-tuning step itself** (the project's Step 3, GPU training with LoRA). That step already runs on a free hosted GPU via **Unsloth's own official Colab/Kaggle notebooks** — see [Unsloth's notebooks page](https://docs.unsloth.ai/get-started/unsloth-notebooks). This course intentionally links to those rather than hosting a duplicate copy, since Unsloth actively maintains them and which model/notebook is current shifts over time (see the lesson's "check current docs" tip).

So the intended flow using this notebook is:

1. Run the dataset-prep cells below to produce `dataset.jsonl`.
2. Download `dataset.jsonl` from this notebook, then go open one of [Unsloth's official notebooks](https://docs.unsloth.ai/get-started/unsloth-notebooks) and fine-tune there, uploading your dataset into *that* notebook.
3. Download the resulting adapter folder from Unsloth's notebook.
4. Come back here (or open a fresh copy of this notebook) and run the inference cells, uploading your adapter, to try your fine-tuned model.

## Setup

Installs the dependencies needed for the local parts only: dataset handling and running inference with a base model + LoRA adapter. (This does *not* install Unsloth itself — that lives in Unsloth's own notebook for the fine-tuning step.)

In [ ]:
!pip install -q datasets huggingface_hub transformers peft torch

## Step 1: Prepare a small dataset

This mirrors `examples/finetune-llm-unsloth/build_dataset.py`. Replace `EXAMPLES` with your own instruction/response pairs for your own capstone project — aim for 30-50+ consistent examples; the handful below is just enough to see the code run.

In [ ]:
import json

EXAMPLES = [
    {
        "instruction": "Summarize this course in one sentence.",
        "response": "A free, browser-based course teaching Python and pandas from first principles through a full data-analysis project.",
    },
    {
        "instruction": "What is a variable in Python?",
        "response": "A variable is a name that points to a value stored in memory, so you can refer to that value again by name instead of retyping it.",
    },
    {
        "instruction": "What is a for loop used for?",
        "response": "A for loop repeats a block of code once for each item in a sequence, like each row of a CSV file or each character in a string.",
    },
    {
        "instruction": "What does pandas' groupby do?",
        "response": "groupby splits a DataFrame into groups based on a column's values, so you can compute a summary statistic (like a mean or count) separately for each group.",
    },
    {
        "instruction": "Why does correlation not imply causation?",
        "response": "Two variables can move together because a third, unmeasured factor drives both of them -- a correlation alone can't rule that out, so it never proves one variable causes the other.",
    },
    {
        "instruction": "What is a LoRA adapter?",
        "response": "A LoRA adapter is a small pair of low-rank matrices trained on top of a frozen pretrained model, letting you specialize the model's behavior without retraining all of its original parameters.",
    },
]

with open("dataset.jsonl", "w", encoding="utf-8") as f:
    for example in EXAMPLES:
        f.write(json.dumps(example) + "\n")

print(f"Wrote {len(EXAMPLES)} examples to dataset.jsonl")

In [ ]:
# Quick sanity check: print the dataset back out.
with open("dataset.jsonl", encoding="utf-8") as f:
    for line in f:
        print(line.strip())

## Now go fine-tune on Unsloth's own notebook

Download `dataset.jsonl` from this environment (in Colab: the Files panel on the left, right-click → Download; in Kaggle: the Output/Data panel), then:

1. Open [Unsloth's official notebooks page](https://docs.unsloth.ai/get-started/unsloth-notebooks) and pick a beginner-friendly notebook for a small (~1B parameter) model.
2. Upload your `dataset.jsonl` into *that* notebook and point its data-loading cell at it instead of its own example dataset.
3. Run that notebook's cells — it fine-tunes the model with LoRA on a free GPU.
4. Download the resulting **adapter** folder (typically tens of megabytes) once training finishes.

Come back here with that adapter folder to try the cells below.

## Step 3: Run your fine-tuned model (local inference)

This mirrors `examples/finetune-llm-unsloth/infer.py`. It loads the base model plus your downloaded LoRA adapter and generates one response.

You need your own adapter folder for this to actually run. In Colab, you can upload it as a zip and unzip it; the cell below tries `google.colab.files.upload()` and falls back to assuming the adapter already exists at `ADAPTER_PATH` (e.g. if you mounted Drive or you're running elsewhere, like Kaggle's own upload panel).

In [ ]:
import zipfile

ADAPTER_PATH = "./my-adapter"

try:
    from google.colab import files

    print("Upload a zip of your downloaded adapter folder (e.g. my-adapter.zip):")
    uploaded = files.upload()
    for filename in uploaded:
        if filename.endswith(".zip"):
            with zipfile.ZipFile(filename, "r") as zf:
                zf.extractall(ADAPTER_PATH)
            print(f"Extracted {filename} to {ADAPTER_PATH}")
except ImportError:
    print(
        "Not running in Colab (or google.colab isn't available). "
        f"Make sure your adapter folder is already present at {ADAPTER_PATH} "
        "(e.g. via Kaggle's upload panel or a mounted drive) before running the next cell."
    )

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE_MODEL_NAME = "unsloth/<the-base-model-you-fine-tuned>"  # match the model used in Unsloth's notebook

PROMPT = "Summarize this course in one sentence."

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME)
base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_NAME)
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)

inputs = tokenizer(PROMPT, return_tensors="pt")
output = model.generate(**inputs, max_new_tokens=80)
print(tokenizer.decode(output[0], skip_special_tokens=True))

## What's next

See the full [Fine-tune a Small LM with Unsloth lesson](https://github.com/abderrahim-lectures/python-data-analysis-course/tree/main/docs/projects/finetune-llm-unsloth) for the complete walkthrough, common pitfalls, and what to try next — including running all of this locally with `uv` instead, which is the recommended path once you're comfortable outside a hosted notebook.